In [10]:
import numpy as np
import pandas as pd
import os
import networkx as nx
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix
from sklearn.utils import shuffle
from sklearn.metrics import classification_report
#from build_graph_data import *
from collections import Counter
from sklearn.manifold import TSNE
import matplotlib.pyplot as plt
import os
import itertools
import numpy as np
import pandas as pd
from itertools import product
from scipy.sparse import coo_matrix
from sklearn.preprocessing import StandardScaler
import torch
import torch.nn.functional as F
from torch_geometric.data import Data
from itertools import product
import numpy as np, torch
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from torch_geometric.utils import to_undirected, add_self_loops

from build_graph_data import *

In [11]:
partition = 100

# Read Data

In [12]:
path = f'../../data/top30groups/noGeographic/combined/combined{partition}.csv'
df = pd.read_csv(path, encoding='ISO-8859-1')
#traindata, valdata, testdata = handle_leakage(df)

In [13]:
df.columns

Index(['extended', 'vicinity', 'multiple', 'success', 'suicide', 'attacktype1',
       'targtype1', 'target1', 'individual', 'weaptype1', 'nkill', 'property',
       'ishostkid', 'gname'],
      dtype='object')

# Create train graph

In [14]:
# Creating df of the feature subset for node creation
#def create_node_dataframe(data, node_features, label_column='gname'):
 #   relevant_data = data[node_features + [label_column]].copy()
#    relevant_data['combination'] = list(zip(*(relevant_data[feat] for feat in node_features)))
 #   df_unique = relevant_data.drop_duplicates(subset=['combination'], keep='first').reset_index(drop=True)
  #  return df_unique[['combination', label_column]]

In [15]:
import torch
import torch.nn.functional as F
from torch_geometric.nn import GCNConv


In [16]:
def induced_subgraph(edge_index, keep_mask):
    keep = keep_mask.nonzero(as_tuple=False).view(-1)
    # map old -> new ids
    new_id = -torch.ones(keep_mask.size(0), dtype=torch.long, device=edge_index.device)
    new_id[keep] = torch.arange(keep.numel(), device=edge_index.device)
    src, dst = edge_index
    m = keep_mask[src] & keep_mask[dst]
    ei = edge_index[:, m]
    ei = torch.stack([new_id[ei[0]], new_id[ei[1]]], dim=0)
    return ei, new_id

In [17]:
import itertools, time, numpy as np, pandas as pd, torch, torch.nn.functional as F
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from torch_geometric.nn import GCNConv

# ----------------------------
# Model with dropout
# ----------------------------
class PyTorchGCN(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels, num_classes, dropout=0.5):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, num_classes)
        self.dropout = torch.nn.Dropout(p=dropout)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = self.dropout(x)
        x = self.conv2(x, edge_index)
        return x

#[[radius k=20 | h= 64 lr=0.01   opt=adam  drop=0.3] VAL f1_ma=0.552 ACC=0.545 | TEST f1_ma=0.471 ACC=0.478
#[knn                k=24 | h=512 lr=0.01   opt=adam  drop=0.3] VAL f1_ma=0.532 ACC=0.537 | TEST f1_ma=0.474 ACC=0.482
#[hybrid_equal_knn   k=23 | h= 32 lr=0.001  opt=adam  drop=0.3] VAL f1_ma=0.517 ACC=0.523 | TEST f1_ma=0.481 ACC=0.487
#[adaptive           k=23 | h= 32 lr=0.01   opt=adam  drop=0.3] VAL f1_ma=0.542 ACC=0.545 | TEST f1_ma=0.482 ACC=0.488
#[knn                k=23 | h=256 lr=0.01   opt=adam  drop=0.3] VAL f1_ma=0.534 ACC=0.540 | TEST f1_ma=0.485 ACC=0.488
#[knn                k=23 | h= 64 lr=0.001  opt=adam  drop=0.5] VAL f1_ma=0.536 ACC=0.543 | TEST f1_ma=0.479 ACC=0.488



# ----------------------------
# Metrics helper
# ----------------------------
def compute_all_metrics(y_true_np, logits_np, n_class):
    """Return dict of acc + P/R/F1 (weighted/micro/macro) + OVR ROC-AUC (w/mi/ma)."""
    y_pred_np = logits_np.argmax(axis=1)
    proba_np  = torch.softmax(torch.from_numpy(logits_np), dim=1).numpy()

    labels = list(range(n_class))
    out = {}
    # Accuracy
    out['acc'] = accuracy_score(y_true_np, y_pred_np)
    # Weighted
    out['prec_w'] = precision_score(y_true_np, y_pred_np, average='weighted', zero_division=0)
    out['rec_w']  = recall_score(  y_true_np, y_pred_np, average='weighted', zero_division=0)
    out['f1_w']   = f1_score(      y_true_np, y_pred_np, average='weighted')
    # Micro
    out['prec_mi'] = precision_score(y_true_np, y_pred_np, average='micro', zero_division=0)
    out['rec_mi']  = recall_score(  y_true_np, y_pred_np, average='micro',   zero_division=0)
    out['f1_mi']   = f1_score(      y_true_np, y_pred_np, average='micro')
    # Macro
    out['prec_ma'] = precision_score(y_true_np, y_pred_np, average='macro', zero_division=0)
    out['rec_ma']  = recall_score(  y_true_np, y_pred_np, average='macro',  zero_division=0)
    out['f1_ma']   = f1_score(      y_true_np, y_pred_np, average='macro')

    # OVR ROC-AUCs (robust if some classes missing in split)
    # If only one class present in y_true, roc_auc_score will raise; guard it.
    def safe_roc(avg):
        try:
            return roc_auc_score(y_true_np, proba_np, multi_class='ovr', average=avg, labels=labels)
        except Exception:
            return float('nan')

    out['roc_w']  = safe_roc('weighted')
    out['roc_mi'] = safe_roc('micro')
    out['roc_ma'] = safe_roc('macro')
    return out

# ----------------------------
# Hyperparameters to sweep
# ----------------------------
hidden_dims   = [16, 32, 64, 128, 256, 512]
lrs           = [0.01, 0.001]
optimizers    = ["adam", "adamw"]
dropouts      = [0.0, 0.3, 0.5]
num_epochs    = 2000
weight_decay  = 5e-4
patience      = 100
seed          = 42
device        = "cuda"

torch.manual_seed(seed); np.random.seed(seed)

# ----------------------------
# Data split (temporal)
# ----------------------------
df_all = df.copy()
train_idx, val_idx, test_idx = temporal_row_split(df_all, label_column='gname', train_ratio=0.6, val_ratio=0.2)

# Graph options
edge_modes = ["hybrid_equal_knn"]
#edge_modes = ["equal"]
k_values   = list(range(7, 13))

node_feature_cols = None
edge_feature_cols = ['attacktype1', 'target1', 'nkill']  # used for 'equal' part; fine to keep for others

def make_optimizer(name, params, lr, wd):
    name = name.lower()
    if name == "adam":  return torch.optim.Adam(params, lr=lr, weight_decay=wd)
    if name == "adamw": return torch.optim.AdamW(params, lr=lr, weight_decay=wd)
    raise ValueError(f"Unknown optimizer: {name}")

def train_one_model(x, edge_index, y, train_mask, val_mask, num_classes,
                    hidden_dim, lr, opt_name, dropout, max_epochs=num_epochs, patience=patience):
    model = PyTorchGCN(in_channels=x.size(1),
                       hidden_channels=hidden_dim,
                       num_classes=num_classes,
                       dropout=dropout).to(device)
    opt = make_optimizer(opt_name, model.parameters(), lr, weight_decay)
    best_state, best_val, patience_left = None, float('inf'), patience

    for epoch in range(1, max_epochs + 1):
        model.train(); opt.zero_grad()
        out = model(x, edge_index)
        loss_train = F.cross_entropy(out[train_mask], y[train_mask])
        loss_train.backward(); opt.step()

        model.eval()
        with torch.no_grad():
            out = model(x, edge_index)
            loss_val = F.cross_entropy(out[val_mask], y[val_mask])

        if loss_val.item() < best_val - 1e-6:
            best_val = loss_val.item()
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience_left = patience
        else:
            patience_left -= 1
            if patience_left <= 0:
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})

    # Validation metrics (selection)
    model.eval()
    with torch.no_grad():
        logits = model(x, edge_index)
    y_val_true = y[val_mask].detach().cpu().numpy()
    y_val_logits = logits[val_mask].detach().cpu().numpy()
    val_metrics = compute_all_metrics(y_val_true, y_val_logits, num_classes)
    val_metrics['val_loss'] = best_val
    return model, val_metrics

def split_metrics(model, x, edge_index, y, mask, num_classes, split_name):
    model.eval()
    with torch.no_grad():
        logits = model(x, edge_index)
    y_true = y[mask].detach().cpu().numpy()
    logits_np = logits[mask].detach().cpu().numpy()
    mets = compute_all_metrics(y_true, logits_np, num_classes)
    return {f"{split_name}_{k}": v for k, v in mets.items()}

# ----------------------------
# Grid search
# ----------------------------
results = []
best_row = None
best_score = -1.0  # we’ll select by val_macro_F1

t_all = time.time()
for k in k_values:
    for edge_mode in edge_modes:
        # A) fit scalers ON TRAIN ONLY
        node_scaler, edge_scaler = fit_scalers(
            df_all,
            node_feature_cols=['extended','vicinity','multiple','success','suicide','attacktype1',
                               'targtype1','target1','individual','weaptype1','nkill','property','ishostkid'],
            edge_feature_cols=edge_feature_cols,
            train_idx=train_idx
        )

        # B) build TRAIN+VAL graph directly (neighbors computed within this set)
        trainval_idx = np.concatenate([train_idx, val_idx])
        x_tv, ei_tv, y_tv, _, _, _, label_index, chosen_radius = build_graph_on_index_set(
            df_all=df_all,
            idx_subset=trainval_idx,
            node_feature_cols=['extended','vicinity','multiple','success','suicide','attacktype1',
                               'targtype1','target1','individual','weaptype1','nkill','property','ishostkid'],
            edge_feature_cols=edge_feature_cols,
            node_scaler=node_scaler, edge_scaler=edge_scaler,
            label_column='gname',
            train_idx=train_idx, val_idx=val_idx, test_idx=None,
            edge_mode=edge_mode, equal_cols=edge_feature_cols, k=k, radius=None  # radius will be auto-picked from TRAIN in this set
        )

        # masks relative to the TRAIN+VAL subgraph:
        # (build them by mapping global indices -> local positions)
        pos_tv = {rid:i for i, rid in enumerate(trainval_idx)}
        train_mask_sub = torch.zeros(x_tv.size(0), dtype=torch.bool, device='cuda')
        val_mask_sub   = torch.zeros_like(train_mask_sub)
        for rid in train_idx: train_mask_sub[pos_tv[rid]] = True
        for rid in val_idx:   val_mask_sub[pos_tv[rid]]   = True

        num_classes = len(label_index)

        for hidden_dim, lr, opt_name, dropout in itertools.product(hidden_dims, lrs, optimizers, dropouts):
            t0 = time.time()
            model, val_m = train_one_model(
                x_tv, ei_tv, y_tv, train_mask_sub, val_mask_sub, num_classes,
                hidden_dim=hidden_dim, lr=lr, opt_name=opt_name, dropout=dropout,
                max_epochs=num_epochs, patience=patience
            )

            # C) build FULL graph (train+val+test) using SAME scalers and SAME edge params
            #    - For radius modes, reuse chosen_radius so ε doesn’t change between train and test graphs.
            x_full, ei_full, y_full, train_mask_full, val_mask_full, test_mask_full, _, _ = build_graph_on_index_set(
                df_all=df_all,
                idx_subset=df_all.index.to_numpy(),  # full
                node_feature_cols=['extended','vicinity','multiple','success','suicide','attacktype1',
                                   'targtype1','target1','individual','weaptype1','nkill','property','ishostkid'],
                edge_feature_cols=edge_feature_cols,
                node_scaler=node_scaler, edge_scaler=edge_scaler,
                label_column='gname',
                train_idx=train_idx, val_idx=val_idx, test_idx=test_idx,
                edge_mode=edge_mode, equal_cols=edge_feature_cols, k=k, radius=chosen_radius
            )

            # TESTA DENNA

            # (Optional stricter inference: one-way edges train→test only)
            ei_infer = make_infer_edge_index_strict(ei_full, train_mask_full | val_mask_full, test_mask_full)
            ei_infer, _ = add_self_loops(ei_infer, num_nodes=x_full.size(0))
            test_m = split_metrics(model, x_full, ei_infer, y_full, test_mask_full, num_classes, "test")

            # Standard transductive inference on full graph:
            #test_m = split_metrics(model, x_full, ei_full, y_full, test_mask_full, num_classes, "test")
            # Test split metrics
            #test_m = split_metrics(model, x, edge_index, y, test_mask, num_classes, split_name="test")
            elapsed = time.time() - t0

            row = dict(
                k=k, edge_mode=edge_mode, hidden_dim=hidden_dim, lr=lr,
                optimizer=opt_name, dropout=dropout, time_sec=elapsed,
                train_nodes=int(train_mask_full.sum().item()),
                val_nodes=int(val_mask_full.sum().item()),
                test_nodes=int(test_mask_full.sum().item()),
            )
            # attach val and test metrics
            for kmet, vmet in val_m.items():
                row[f"val_{kmet}"] = vmet
            row.update(test_m)

            results.append(row)

            # Selection criterion: validation macro-F1
            sel = val_m['f1_ma']
            if sel > best_score:
                best_score = sel
                best_row = row.copy()

            print(f"[{edge_mode:18s} k={k:2d} | h={hidden_dim:3d} lr={lr:<6} opt={opt_name:<5} drop={dropout}] "
                  f"VAL f1_ma={val_m['f1_ma']:.3f} ACC={val_m['acc']:.3f} | "
                  f"TEST f1_ma={test_m['test_f1_ma']:.3f} ACC={test_m['test_acc']:.3f}")

total_min = (time.time() - t_all)/60
print(f"\nGrid search finished in {total_min:.1f} min")

#results_df = pd.DataFrame(results)

# ----------------------------
# Save all runs + best summary
# ----------------------------
#results_csv = f"grid_search_results{partition}.csv"
#results_df.to_csv(results_csv, index=False)

best_txt = f"best_run_{partition}.txt"
with open(best_txt, "w") as f:
    f.write("=== Best configuration (by validation macro-F1) ===\n")
    f.write(f"edge_mode   : {best_row['edge_mode']}\n")
    f.write(f"k           : {best_row['k']}\n")
    f.write(f"hidden_dim  : {best_row['hidden_dim']}\n")
    f.write(f"optimizer   : {best_row['optimizer']}\n")
    f.write(f"lr          : {best_row['lr']}\n")
    f.write(f"dropout     : {best_row['dropout']}\n")
    f.write("\n--- Validation metrics ---\n")
    f.write(f"val_loss    : {best_row.get('val_val_loss', float('nan')):.6f}\n")
    f.write(f"val_acc     : {best_row.get('val_acc', float('nan')):.6f}\n")
    f.write(f"val_prec_w  : {best_row.get('val_prec_w', float('nan')):.6f}\n")
    f.write(f"val_rec_w   : {best_row.get('val_rec_w', float('nan')):.6f}\n")
    f.write(f"val_f1_w    : {best_row.get('val_f1_w', float('nan')):.6f}\n")
    f.write(f"val_prec_mi : {best_row.get('val_prec_mi', float('nan')):.6f}\n")
    f.write(f"val_rec_mi  : {best_row.get('val_rec_mi', float('nan')):.6f}\n")
    f.write(f"val_f1_mi   : {best_row.get('val_f1_mi', float('nan')):.6f}\n")
    f.write(f"val_prec_ma : {best_row.get('val_prec_ma', float('nan')):.6f}\n")
    f.write(f"val_rec_ma  : {best_row.get('val_rec_ma', float('nan')):.6f}\n")
    f.write(f"val_f1_ma   : {best_row.get('val_f1_ma', float('nan')):.6f}\n")
    f.write(f"val_roc_w   : {best_row.get('val_roc_w', float('nan')):.6f}\n")
    f.write(f"val_roc_mi  : {best_row.get('val_roc_mi', float('nan')):.6f}\n")
    f.write(f"val_roc_ma  : {best_row.get('val_roc_ma', float('nan')):.6f}\n")
    f.write("\n--- Test metrics ---\n")
    f.write(f"test_acc    : {best_row.get('test_acc', float('nan')):.6f}\n")
    f.write(f"test_prec_w : {best_row.get('test_prec_w', float('nan')):.6f}\n")
    f.write(f"test_rec_w  : {best_row.get('test_rec_w', float('nan')):.6f}\n")
    f.write(f"test_f1_w   : {best_row.get('test_f1_w', float('nan')):.6f}\n")
    f.write(f"test_prec_mi: {best_row.get('test_prec_mi', float('nan')):.6f}\n")
    f.write(f"test_rec_mi : {best_row.get('test_rec_mi', float('nan')):.6f}\n")
    f.write(f"test_f1_mi  : {best_row.get('test_f1_mi', float('nan')):.6f}\n")
    f.write(f"test_prec_ma: {best_row.get('test_prec_ma', float('nan')):.6f}\n")
    f.write(f"test_rec_ma : {best_row.get('test_rec_ma', float('nan')):.6f}\n")
    f.write(f"test_f1_ma  : {best_row.get('test_f1_ma', float('nan')):.6f}\n")
    f.write(f"test_roc_w  : {best_row.get('test_roc_w', float('nan')):.6f}\n")
    f.write(f"test_roc_mi : {best_row.get('test_roc_mi', float('nan')):.6f}\n")
    f.write(f"test_roc_ma : {best_row.get('test_roc_ma', float('nan')):.6f}\n")
    f.write("\n--- Data sizes ---\n")
    f.write(f"train_nodes : {best_row['train_nodes']}\n")
    f.write(f"val_nodes   : {best_row['val_nodes']}\n")
    f.write(f"test_nodes  : {best_row['test_nodes']}\n")
    #f.write(f"\nCSV of all runs: {results_csv}\n")

#print(f"Saved all runs to {results_csv} and best summary to {best_txt}")


[hybrid_equal_knn   k= 7 | h= 16 lr=0.01   opt=adam  drop=0.0] VAL f1_ma=0.422 ACC=0.435 | TEST f1_ma=0.335 ACC=0.340
[hybrid_equal_knn   k= 7 | h= 16 lr=0.01   opt=adam  drop=0.3] VAL f1_ma=0.423 ACC=0.442 | TEST f1_ma=0.325 ACC=0.338
[hybrid_equal_knn   k= 7 | h= 16 lr=0.01   opt=adam  drop=0.5] VAL f1_ma=0.432 ACC=0.450 | TEST f1_ma=0.335 ACC=0.343
[hybrid_equal_knn   k= 7 | h= 16 lr=0.01   opt=adamw drop=0.0] VAL f1_ma=0.404 ACC=0.413 | TEST f1_ma=0.301 ACC=0.312
[hybrid_equal_knn   k= 7 | h= 16 lr=0.01   opt=adamw drop=0.3] VAL f1_ma=0.416 ACC=0.427 | TEST f1_ma=0.300 ACC=0.312
[hybrid_equal_knn   k= 7 | h= 16 lr=0.01   opt=adamw drop=0.5] VAL f1_ma=0.426 ACC=0.437 | TEST f1_ma=0.331 ACC=0.340
[hybrid_equal_knn   k= 7 | h= 16 lr=0.001  opt=adam  drop=0.0] VAL f1_ma=0.403 ACC=0.418 | TEST f1_ma=0.274 ACC=0.283
[hybrid_equal_knn   k= 7 | h= 16 lr=0.001  opt=adam  drop=0.3] VAL f1_ma=0.409 ACC=0.420 | TEST f1_ma=0.343 ACC=0.350
[hybrid_equal_knn   k= 7 | h= 16 lr=0.001  opt=adam  dro

In [18]:
test_m

{'test_acc': 0.38166666666666665,
 'test_prec_w': 0.3869086036419988,
 'test_rec_w': 0.38166666666666665,
 'test_f1_w': 0.373646882953618,
 'test_prec_mi': 0.38166666666666665,
 'test_rec_mi': 0.38166666666666665,
 'test_f1_mi': 0.38166666666666665,
 'test_prec_ma': 0.38690860364199875,
 'test_rec_ma': 0.3816666666666667,
 'test_f1_ma': 0.37364688295361803,
 'test_roc_w': 0.8683836206896552,
 'test_roc_mi': 0.8722291666666667,
 'test_roc_ma': 0.8683836206896554}

In [19]:
print(classification_report(y_true, y_pred))

NameError: name 'y_true' is not defined